# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 11: Production Deployment Patterns: Shadowing vs Canary Routing

### 1. Conceptual Intuition & Theoretical Background
Deploying new ML models directly to 100% of live traffic risks system outages, latency spikes, and unexpected prediction errors.

**1. Shadow Deployment (Dark Launching):**
Incoming production traffic is mirrored asynchronously to the candidate model (Model B) alongside the production model (Model A):
- Model A returns live responses to the end-user.
- Model B executes asynchronously in the background.
- Zero customer impact; tests real-world latency, throughput, and inference drift safely.

**2. Canary Routing:**
Routes a small fraction (e.g., $5\%$) of live traffic to Model B. If error rates, latencies, or business KPIs stay within SLA bounds, traffic ramps progressively ($5\% \to 25\% \to 100\%$). Automated circuit breakers trigger instant rollbacks on SLA breaches.

---

### 2. Scratch Python Implementation: Traffic Routing & Circuit Breaker Engine

In [ ]:
import numpy as np

class ProductionModelRouter:
    def __init__(self, canary_percentage=0.05, latency_sla_ms=40.0):
        self.canary_pct = canary_percentage
        self.sla_ms = latency_sla_ms
        self.tripped_circuit_breaker = False
        
        # Telemetry logs
        self.latency_log = []
        self.routed_targets = []
        
    def model_a_champion(self, req):
        latency = np.random.normal(15.0, 2.0)
        return "champion_pred", latency
        
    def model_b_candidate(self, req):
        # Candidate model has latency spike
        latency = np.random.normal(32.0, 6.0)
        return "candidate_pred", latency
        
    def route_request(self, req):
        # Fall back to Champion if circuit breaker is tripped
        if self.tripped_circuit_breaker:
            pred, lat = self.model_a_champion(req)
            self.routed_targets.append('CHAMPION_FALLBACK')
            return pred
            
        # Canary probabilistic routing
        is_canary = np.random.rand() < self.canary_pct
        
        if is_canary:
            pred, lat = self.model_b_candidate(req)
            self.routed_targets.append('CANARY_CANDIDATE')
            self.latency_log.append(lat)
            
            # Circuit breaker check: trip if p95 latency breaches SLA
            if len(self.latency_log) >= 10:
                p95_lat = np.percentile(self.latency_log, 95)
                if p95_lat > self.sla_ms:
                    print(f"  [ALERT] Circuit Breaker TRIPPED! P95 Latency ({p95_lat:.1f}ms) breached SLA ({self.sla_ms}ms)!")
                    self.tripped_circuit_breaker = True
            return pred
        else:
            pred, lat = self.model_a_champion(req)
            self.routed_targets.append('CHAMPION')
            return pred

# Run 200 Production Requests through Canary Router
np.random.seed(42)
router = ProductionModelRouter(canary_percentage=0.15, latency_sla_ms=38.0)

for req_id in range(200):
    router.route_request(f"req_{req_id}")

counts = {t: router.routed_targets.count(t) for t in set(router.routed_targets)}
print("--- Canary Routing Telemetry ---")
print(f"Traffic Distribution: {counts}")
print(f"Circuit Breaker Status: {'TRIPPED' if router.tripped_circuit_breaker else 'HEALTHY'}")

### 3. Step-by-Step Code Walkthrough
1. **Canary Gating:** Evaluates uniform random variables to split off a $15\%$ traffic slice.
2. **SLA Monitoring:** Evaluates rolling 95th percentile latency against defined bounds.
3. **Automated Circuit Breaker:** Reverts traffic to champion model upon SLA breach.

---

### 4. Industry Applications & Trade-offs
- **High-Scale Serving (Netflix, Uber Michelangelo):** Testing new recommendation algorithms on internal employee traffic before customer rollout.
- **Trade-off:** Shadow deployments double backend compute costs because every request executes twice.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Blue-Green Deployment and Canary Deployment?**
> **Answer:** Blue-Green runs two identical production environments and switches 100% of traffic all at once via router cutover. Canary routes a small percentage of users incrementally, validating performance before full cutover.

**Q2: What is the primary advantage of Shadow Deployment over Canary Deployment?**
> **Answer:** Shadow deployment carries zero risk of impacting end-users; candidate model failures or corrupt outputs remain invisible to customers.